# 001 · 把时间放进算符

跟做官方QobjEvo教程的核心任务。先运行每格，再改frequency，预测变化。原文在third_party/qutip_tutorials/001_qobjevo.md；作者、许可证与固定版本见docs/sources.json。

Qobj是某个固定算符；QobjEvo像一个“给定时间就返回算符”的函数。

In [1]:
from pathlib import Path
import sys
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
from IPython.display import display
root = Path.cwd()
if not (root / 'code').is_dir():
    root = root.parent
if not (root / 'code').is_dir():
    raise RuntimeError('请在项目根目录或notebooks目录运行')
sys.path.insert(0, str(root / 'code'))
print('当前解释器：', sys.executable)
options = dict(atol=1e-11, rtol=1e-9, method='dop853', store_states=True, normalize_output=False)


当前解释器： D:\Master.s-work\Software-needed-for-master.s-work\python\main\python.exe


## 三种系数表示同一H(t)
H(t)=a†a+cos(ωt)(a+a†)。数组系数在点之间需要插值；字符串使用解释执行，省去编译器。

In [2]:
from qutip.core.coefficient import CompilationOptions
a = qt.destroy(4)
number, quadrature = a.dag()*a, a+a.dag()
def drive(t, frequency):
    return np.cos(frequency*t)
function = qt.QobjEvo([number, [quadrature, drive]], args={'frequency':1.0})
string_coefficient = qt.coefficient('cos(frequency*t)', args={'frequency':1.0}, compile_opt=CompilationOptions(use_cython=False))
string = number + quadrature*string_coefficient
times = np.linspace(0,10,1001)
array = qt.QobjEvo([number,[quadrature,np.cos(times)]], tlist=times)
test_time = 0.137
print('函数和字符串的矩阵误差：', np.max(abs(function(test_time).full()-string(test_time).full())))
print('数组插值的矩阵误差：', np.max(abs(array(test_time).full()-function(test_time).full())))
np.testing.assert_allclose(function(test_time).full(), array(test_time).full(), atol=1e-7, rtol=0)


D:\Master.s-work\Software-needed-for-master.s-work\python\main\Lib\site-packages\qutip\core\coefficient.py:426: UserWarning: `cython`, `setuptools` and `filelock` are required for compilation of string coefficents. Falling back on `eval`.
  warnings.warn(
函数和字符串的矩阵误差： 0.0
数组插值的矩阵误差： 3.154720928932875e-11


## 改参数与量子对象
临时传入frequency不应永久改变初始参数。再看一次shape与dims，练习dag、tensor、expect。

In [3]:
changed = function(0.7, frequency=2.0)
expected = number+np.cos(1.4)*quadrature
np.testing.assert_allclose(changed.full(),expected.full(),atol=1e-12)
np.testing.assert_allclose(function(0.7).full(),(number+np.cos(0.7)*quadrature).full(),atol=1e-12)
joint = qt.tensor(qt.basis(4,2),qt.basis(2,0))
observable = qt.tensor(number,qt.qeye(2))
print('shape:',joint.shape,'dims:',joint.dims,'模式光子数:',qt.expect(observable,joint))
liouvillian = qt.liouvillian(function,c_ops=[0.1*a])
print('Liouvillian shape:',liouvillian(0.7).shape)


shape: (8, 1) dims: [[4, 2], [1]] 模式光子数: 2.0000000000000004
Liouvillian shape: (16, 16)


## 自己改一处
把频率改成2，数组也必须改成cos(2*times)。然后尝试把数组采样降到21个点，检查非节点处的误差是否变大。不要只在已有采样点比较。